# 32. 6조건 검색·답변 평가

OCR 통과 가정. 원본은 읽기 전용이며 현재 Upstage 계층을 수정하지 않습니다. Python 실행은 conda skn25만 허용합니다.

## 재현과 책임

장시간 API·모델 실행은 동일 payload 캐시·체크포인트·실패 기록을 가진 runner가 담당합니다. 이 노트북의 셀은 저장된 실제 결과를 읽고 보여주며 보고서를 생성합니다. 유료 작업을 셀에서 처음부터 재실행하지 않습니다.

아래 각 Python 명령 앞에 `PYTHONDONTWRITEBYTECODE=1 conda run --no-capture-output -n skn25`를 붙입니다. 원본 OCR lane·코드, 동결된 JSON/SQLite·청크·새 인덱스, API 캐시와 입력 hash가 필요합니다. 입력 hash가 바뀌면 캐시를 자동 덮어쓰지 않고 중단합니다. 캐시가 없는 API 실행에는 키·네트워크·비용이 필요합니다. 원본 데이터와 API 키는 Git 커밋에 포함하지 않습니다. DB 쓰기는 database-manager가 담당합니다.

```bash
# 기존 dataset_freeze.json은 동결된 입력입니다. 재현 시 임의 재주석하지 않습니다.
python -B notebooks/32_search_experiment.py retrieval
python -B notebooks/32_search_experiment.py answers
python -B notebooks/32_score_answers.py --source oracle
python -B notebooks/32_failure_diagnostics.py
python -B notebooks/32_backend_scope_probe.py
```

모든 결과가 완료된 뒤 `30_summarize_experiments.py`, `30_execute_experiment_notebooks.py`, `30_render_experiment_reports.py`, `30_verify_preservation.py` 순서로 최종 산출물을 확인합니다.

In [1]:
from pathlib import Path
import importlib.util, sys, json
root=Path.cwd()
if root.name=='notebooks': root=root.parent
spec=importlib.util.spec_from_file_location('structured_common',root/'notebooks/30_structured_search_common.py')
C=importlib.util.module_from_spec(spec);sys.modules[spec.name]=C;spec.loader.exec_module(C)
assert Path(sys.prefix).name=='skn25'
print({'environment':'skn25','model':C.MODEL,'reasoning':C.REASONING,'originals':'read only'})

{'environment': 'skn25', 'model': 'gpt-6-luna', 'reasoning': 'medium', 'originals': 'read only'}


In [2]:
files=['dataset_freeze.json', 'retrieval_results.json', 'answer_results.json']
for name in files:
    path=C.STAGES[32]/name
    if not path.exists(): raise RuntimeError('Required stage artifact not complete: '+name)
    data=C.read_json(path)
    if 'records' in data:
        print(name,{'records':len(data['records']),'status_counts':{s:sum(r['status']==s for r in data['records']) for s in ['ok','failed']}})
    elif 'queries' in data: print(name,{'queries':len(data['queries']),'sha256':data['sha256']})
    else: print(json.dumps(data,ensure_ascii=False,indent=2)[:15000])

dataset_freeze.json {'queries': 33, 'sha256': '201c37c52d7a9042be2892b9557d0a1e157c9dd7e9b9a2ab445b7c80d8e6cc18'}
retrieval_results.json {'records': 198, 'status_counts': {'ok': 194, 'failed': 4}}
answer_results.json {'records': 198, 'status_counts': {'ok': 189, 'failed': 9}}


## 해석 범위

source-relative raw OCR 필드 라벨과 새로운 개발/확증 cohort입니다. 독립 PDF 검증, 전수 정답 recall, 운영 채택을 뜻하지 않습니다. 프로필 비교는 청킹·재정렬·근거 구성 전체의 비교입니다. 완전한 재현은 입력 hash·원본 코드 hash·새 인덱스·저장 API 응답과 보고서의 실패 분모를 함께 확인해야 합니다.

In [3]:
spec=importlib.util.spec_from_file_location('experiment_reports',root/'notebooks/30_render_experiment_reports.py')
R=importlib.util.module_from_spec(spec);sys.modules[spec.name]=R;spec.loader.exec_module(R)
print(R.render(32).relative_to(root))

docs/report/32_structured_search_comparison_report.html


In [4]:
freeze=C.read_json(C.STAGES[32]/'dataset_freeze.json')
records=C.read_json(C.STAGES[32]/'retrieval_results.json')['records']
print(json.dumps(R.aggregate(records,{q['query_id']:q for q in freeze['queries']}),ensure_ascii=False,indent=2))
for name in ['answer_constraint_scores.json','failure_diagnostics.json','backend_probe_results.json']:
    data=C.read_json(C.STAGES[32]/name)
    print(name,json.dumps(data if 'records' not in data else {'records':len(data['records'])},ensure_ascii=False)[:10000])

[
  {
    "profile": "card_page_section_benefit",
    "mode": "S_PLUS_U",
    "attempted": 33,
    "ok": 33,
    "failed": 0,
    "positive_denominator": 27,
    "witness_denominator": 24,
    "hit": 1.0,
    "mrr": 1.0,
    "recall": 0.5079398996065663,
    "witness": 0.3473939255189255,
    "fallback": 8,
    "wall": 0.5445688319923647
  },
  {
    "profile": "card_page_section_benefit",
    "mode": "S_THEN_U",
    "attempted": 33,
    "ok": 33,
    "failed": 0,
    "positive_denominator": 27,
    "witness_denominator": 24,
    "hit": 1.0,
    "mrr": 1.0,
    "recall": 0.5654626237959571,
    "witness": 0.40813873626373626,
    "fallback": 8,
    "wall": 2.1437762700115854
  },
  {
    "profile": "card_page_section_benefit",
    "mode": "U",
    "attempted": 33,
    "ok": 33,
    "failed": 0,
    "positive_denominator": 27,
    "witness_denominator": 24,
    "hit": 0.9629629629629629,
    "mrr": 0.9444444444444444,
    "recall": 0.3881596798263465,
    "witness": 0.26584630647130647,